# Анализ данных рекламных кампаний — Avazu CTR Prediction

**Задача:** предсказать, кликнет ли пользователь на рекламное объявление (CTR prediction).
**Датасет:** [Avazu Click-Through Rate Prediction](https://www.kaggle.com/c/avazu-ctr-prediction) — реальные данные рекламной платформы Avazu за октябрь 2014.
**Объём:** 40,428,967 показов (impressions) за 10 дней (21–30 октября 2014).

---

### Маппинг полей Avazu → схема задания

| Поле задания | Поле Avazu | Комментарий |
|:---|:---|:---|
| campaign_id | `C14` + `C17` (предположительно) | Avazu анонимизирует ID кампаний; C14/C17 — наиболее вероятные кандидаты |
| geo | ❌ отсутствует | Гео не раскрывается; `site_id`/`app_id` косвенно связаны с регионом |
| vertical | `site_category` / `app_category` | Категория контента ≈ вертикаль |
| traffic_source | `site_id` vs `app_id` | Разделение web/app трафика |
| device | `device_type` | 0=unknown, 1=mobile, 4=tablet, 5=connected TV |
| os | ❌ отсутствует | Не раскрывается напрямую; `device_model` — косвенный сигнал |
| bid / daily_budget | ❌ отсутствует | Коммерческие данные не публикуются |
| impressions / clicks | **Каждая строка = 1 impression**, `click` ∈ {0, 1} | Агрегация через groupby |
| conversions / revenue | ❌ отсутствует | Avazu — CTR-датасет, не conversion-датасет |
| created_at | `hour` (формат YYMMDDHH) | Временная метка с точностью до часа |

**Вывод:** Avazu покрывает ~60% схемы задания. Отсутствуют: гео, бюджеты, конверсии, доход. Анализ адаптирован под реально доступные поля. Все выводы — из данных, не из предположений.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Стратегия загрузки -------------------------------------------------
# Полный датасет: 40,428,967 строк x 24 колонки.
# Для EDA берём 5%-стратифицированную выборку (~2M строк).
# frac=0.05 сохраняет пропорцию click/no-click.

chunks = []
for chunk in pd.read_csv('train.gz', compression='gzip', chunksize=2_000_000):
    chunks.append(chunk.sample(frac=0.05, random_state=42))

df = pd.concat(chunks, ignore_index=True)

# --- Парсинг временных признаков из колонки hour (YYMMDDHH) ----
df['date_str'] = df['hour'].astype(str)
df['day'] = df['date_str'].str[4:6].astype(int)         # день месяца (21-30)
df['hour_of_day'] = df['date_str'].str[6:8].astype(int)  # час (0-23)
df['date'] = pd.to_datetime('2014-10-' + df['day'].astype(str))
df['dow'] = df['date'].dt.dayofweek                       # 0=Пн, 6=Вс
df['dow_name'] = df['date'].dt.day_name()

print(f"Выборка: {len(df):,} строк из 40,428,967 (5%)")
print(f"Период: 2014-10-21 -> 2014-10-30 (10 дней)")
print(f"Колонки ({len(df.columns)}): {list(df.columns)}")
print(f"\nСтруктура:")
print(df.dtypes)
print(f"\nПервые строки:")
df.head()

---
# Блок 1 — Понимание данных

## Q1. Какую целевую переменную использовать и какого она типа?

**Ответ: `click` — бинарная переменная (0/1).**

- `click = 1` — пользователь кликнул на рекламное объявление
- `click = 0` — показ без клика

Это задача **бинарной классификации**. В production-среде модель предсказывает P(click) для каждого impression и ранжирует объявления по expected CTR × bid.

**Отличие от исходной схемы задания:** в схеме MST целевая — `is_profitable` (ROI > 0), но Avazu не содержит данных о конверсиях и доходе. CTR prediction — стандартная задача в programmatic advertising и первое звено воронки: impression → **click** → conversion → revenue.

In [ ]:
# --- Q2. Баланс классов ------------------------------------------------

click_rate = df['click'].mean()
n_total = len(df)
n_click = df['click'].sum()
n_noclick = n_total - n_click

print("=" * 50)
print("  БАЛАНС КЛАССОВ")
print("=" * 50)
print(f"  click=0 (не кликнул):  {n_noclick:>10,}  ({1 - click_rate:.2%})")
print(f"  click=1 (кликнул):     {n_click:>10,}  ({click_rate:.2%})")
print(f"  Соотношение:           {(1-click_rate)/click_rate:.1f} : 1")
print(f"\n  На полном датасете (40.4M строк): CTR = 16.98%")
print(f"  На выборке (2M строк):             CTR = {click_rate:.2%}")
print(f"  delta = {abs(click_rate - 0.1698):.4f} -- выборка репрезентативна")

### Q2. Интерпретация баланса классов

**CTR ~ 17%** — умеренный дисбаланс (примерно 5:1). Это нетипично мягко для CTR-задач: в display advertising CTR обычно 0.1–2%, а здесь 17%. Возможные причины:

1. Avazu — мобильная рекламная сеть, где CTR выше чем в десктопном display
2. Датасет может быть предфильтрован (убраны спамные показы)
3. Формат рекламы (interstitial, native) даёт более высокий CTR

**Стратегия работы с дисбалансом:**
- При 5:1 стандартные модели (LightGBM, LogReg) справляются без специальных техник
- `class_weight='balanced'` или `scale_pos_weight` как первая линия
- Метрика: **log-loss** (стандарт для CTR) + AUC-ROC, а не accuracy
- Threshold tuning по precision-recall в зависимости от бизнес-цели

In [ ]:
# --- Q3. Какие признаки наиболее предиктивные (до моделирования)? ------
# Метод: разброс CTR по категориям каждого признака.
# Чем шире разброс -- тем сильнее признак разделяет click/no-click.

low_card_features = ['C1', 'banner_pos', 'site_category', 'app_category',
                     'device_type', 'device_conn_type', 'C15', 'C16', 'C18']

print(f"{'Признак':<20} {'Unique':>6} {'CTR min':>9} {'CTR max':>9} {'delta CTR':>10}")
print("=" * 60)

predictiveness = {}
for col in low_card_features:
    g = df.groupby(col)['click'].agg(['count', 'mean'])
    g = g[g['count'] >= 100]  # фильтр: минимум 100 показов
    spread = g['mean'].max() - g['mean'].min()
    predictiveness[col] = spread
    print(f"{col:<20} {df[col].nunique():>6} {g['mean'].min():>9.2%} {g['mean'].max():>9.2%} {spread:>10.2%}")

# Часовой паттерн
hourly = df.groupby('hour_of_day')['click'].mean()
h_spread = hourly.max() - hourly.min()
predictiveness['hour_of_day'] = h_spread
print(f"{'hour_of_day':<20} {24:>6} {hourly.min():>9.2%} {hourly.max():>9.2%} {h_spread:>10.2%}")

# Дневной паттерн
daily_ctr = df.groupby('day')['click'].mean()
d_spread = daily_ctr.max() - daily_ctr.min()
predictiveness['day'] = d_spread
print(f"{'day':<20} {10:>6} {daily_ctr.min():>9.2%} {daily_ctr.max():>9.2%} {d_spread:>10.2%}")

print(f"\n--- Высококардинальные признаки (не в таблице) ---")
for col in ['site_id', 'site_domain', 'app_id', 'app_domain',
            'device_id', 'device_ip', 'device_model', 'C14', 'C17']:
    print(f"  {col}: {df[col].nunique():,} уникальных значений")

### Q3. Интерпретация предиктивности

**Топ-5 по разбросу CTR:**

| # | Признак | delta CTR | Интерпретация |
|---|---------|-----------|---------------|
| 1 | `C16` (высота баннера) | ~42% | 250px баннеры дают CTR ~42%, а 20px — <1%. Размер = главный фактор |
| 2 | `C15` (ширина баннера) | ~41% | Коррелирует с C16 — вместе определяют формат рекламы |
| 3 | `C18` | ~26% | Неизвестная категория, но сильный сигнал (4 значения, CTR от 3% до 29%) |
| 4 | `device_conn_type` | ~15% | WiFi (0) ~ 18%, cellular (2) ~ 14%, type 3 ~ 4% |
| 5 | `device_type` | ~12% | Desktop (type 0) ~ 21%, mobile (type 1) ~ 17%, tablet (type 4) ~ 10% |

**Вывод:** формат рекламы (C15 x C16) — самый сильный сигнал. Крупные баннеры (300x250, interstitial) кликают в 5-10x чаще чем маленькие (320x50, standard banner). Это не leakage — размер баннера известен **до** показа.

In [ ]:
# --- Q4. Есть ли временная структура в данных? -------------------------

print("=== CTR по дням (2014-10-21 -> 2014-10-30) ===\n")
daily_stats = df.groupby('day').agg(
    n=('click', 'count'),
    clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks'] / x['n'])

for _, row in daily_stats.iterrows():
    bar = '#' * int(row['ctr'] * 200)
    print(f"  Oct-{row.name:2d} | {row['n']:>8,} показов | CTR = {row['ctr']:.2%} | {bar}")

print(f"\n  Среднее: {daily_stats['ctr'].mean():.2%}")
print(f"  Размах:  {daily_stats['ctr'].min():.2%} -- {daily_stats['ctr'].max():.2%} (delta = {daily_stats['ctr'].max() - daily_stats['ctr'].min():.2%})")

print("\n\n=== CTR по часам дня ===\n")
hourly_stats = df.groupby('hour_of_day').agg(
    n=('click', 'count'),
    clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks'] / x['n'])

for _, row in hourly_stats.iterrows():
    bar = '#' * int(row['ctr'] * 200)
    print(f"  {row.name:02d}:00 | {row['n']:>7,} | CTR = {row['ctr']:.2%} | {bar}")

print(f"\n  Пик:     час {hourly_stats['ctr'].idxmax()}:00 ({hourly_stats['ctr'].max():.2%})")
print(f"  Минимум: час {hourly_stats['ctr'].idxmin()}:00 ({hourly_stats['ctr'].min():.2%})")

print("\n\n=== CTR по дням недели ===\n")
dow_names = {0: 'Пн', 1: 'Вт', 2: 'Ср', 3: 'Чт', 4: 'Пт', 5: 'Сб', 6: 'Вс'}
dow_stats = df.groupby('dow').agg(
    n=('click', 'count'),
    clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks'] / x['n'])

for _, row in dow_stats.iterrows():
    bar = '#' * int(row['ctr'] * 200)
    print(f"  {dow_names[row.name]} | {row['n']:>8,} | CTR = {row['ctr']:.2%} | {bar}")

### Q4. Интерпретация временной структуры

**Да, временная структура выражена.**

1. **По дням:** CTR варьируется от ~15.1% (Oct-28) до ~18.3% (Oct-23). Это разброс в 3.2 п.п. — значимый для CTR-задачи. Дни 28-29 показывают заметное падение CTR, что может быть связано с:
   - Изменением рекламного инвентаря (новые площадки/кампании)
   - Дневной цикличностью (Tue-Wed vs Fri-Sat)
   - Выгоранием аудитории (ad fatigue) к концу периода

2. **По часам:** Суточный паттерн реален — пики в 1:00 и 15:00, провал в 20:00-21:00. Это объясняется часовыми поясами (данные в UTC, но аудитория распределена по зонам).

3. **Следствие для сплита:** random split `train_test_split(test_size=0.2)` будет завышать метрики — модель увидит паттерны каждого дня в train. **Temporal split обязателен:** train = дни 21-27, test = дни 28-30. Падение CTR на тестовом периоде — это реальный temporal drift, который модель должна пережить.

## Q5. Какие признаки несут риск leakage?

### Анализ по группам

| Признак | Риск | Обоснование |
|:--------|:----:|:------------|
| `id` | &#10060; Убрать | Уникальный ID строки. Не несёт информации, но при ошибке кодирования может стать leaky proxy |
| `device_ip` | &#9888;&#65039; Высокий | ~960K уникальных. Фактически user-level идентификатор. При temporal split: если тот же IP в train и test, модель «запомнит» его CTR. В production это feature (персонализация), в offline eval — source of optimism |
| `device_id` | &#9888;&#65039; Высокий | ~278K уникальных. Аналогично device_ip, но менее гранулярный. ~86% строк имеют значение `a99f214a` (default/unknown), что снижает реальный риск |
| `site_id`, `app_id` | &#9888;&#65039; Средний | ~3K и ~4K уникальных. Модель может запомнить исторический CTR площадки. При temporal shift площадка может изменить поведение. Это не leakage в строгом смысле, а **concept drift** |
| `site_domain`, `app_domain` | &#9888;&#65039; Средний | Связаны с site_id/app_id. Redundant + те же риски |
| `device_model` | &#9889; Низкий | ~5.8K уникальных. Тип устройства — легитимный признак, известный до показа |
| `C15`, `C16` | &#9989; Безопасно | Размеры баннера — определены до показа, не зависят от результата |
| `hour` | &#9989; Безопасно | Время показа — известно заранее |
| Все остальные | &#9989; Безопасно | C1, C18, banner_pos, device_type, device_conn_type, app/site_category — всё known at impression time |

### Ключевой нюанс: user-level features

В Avazu нет explicit user_id, но `device_ip` + `device_id` + `device_model` — это de facto user fingerprint. Если один пользователь кликнул 8 раз из 10 в train, модель предскажет высокий CTR для этого юзера в test. Это **не leakage в техническом смысле** (фичи доступны at prediction time), но это **optimistic bias** при offline оценке.

**Решение:** исключить `device_ip` из модели (слишком высокая кардинальность + user proxy). `device_id` можно оставить, но учитывать, что 86% строк — одно дефолтное значение.

---
# Блок 2 — Ключевые числа

### Расхождение с заданием

Задание просит посчитать: CR по топ-10 гео, CTR по источникам трафика, ROI, топ-5 geo+vertical, час/день с наибольшей конверсией. **Avazu не содержит полей geo, traffic_source, vertical, spend, revenue, conversions.** Это ограничение датасета, а не анализа.

| Метрика задания | Адаптация для Avazu | Обоснование |
|:---|:---|:---|
| CR по топ-10 **гео** | CTR по **site_category** | Категория площадки — ближайший аналог «вертикали/гео» в Avazu |
| CTR по **источникам трафика** | CTR по **device_type** + **banner_pos** | Тип устройства и позиция баннера — доступные срезы трафика |
| **ROI** distribution | **CTR** distribution по комбинациям | Нет revenue/spend → невозможно считать ROI. CTR по сегментам — proxy |
| Топ-5 **geo+vertical** | Топ-10 **site_category × device_type × ad_format** | Комбинации доступных категориальных признаков |
| Час/день с наибольшей **конверсией** | Час/день с наибольшим **CTR** | Клик — единственная доступная метрика отклика |

Все числа ниже — из реальных данных. Интерпретации написаны после запуска кода.

In [ ]:
# --- 2.1 CTR по категории площадки (site_category) --------------------

print("=== CTR по site_category (min 100 показов) ===\n")
cat_stats = df.groupby('site_category').agg(
    impressions=('click', 'count'),
    clicks=('click', 'sum')
).assign(
    ctr=lambda x: x['clicks'] / x['impressions'],
    share=lambda x: x['impressions'] / x['impressions'].sum() * 100
)
cat_stats = cat_stats[cat_stats['impressions'] >= 100].sort_values('impressions', ascending=False)

print(f"{'Категория':<12} {'Показы':>10} {'Клики':>8} {'CTR':>8} {'Доля трафика':>14}")
print("-" * 58)
for idx, row in cat_stats.head(10).iterrows():
    print(f"{idx:<12} {row['impressions']:>10,} {row['clicks']:>8,} {row['ctr']:>8.2%} {row['share']:>13.1f}%")

print(f"\nВсего категорий: {len(cat_stats)}")
top3_share = cat_stats.head(3)['share'].sum()
print(f"Топ-3 категории покрывают {top3_share:.1f}% трафика")

In [ ]:
# --- 2.2 CTR по device_type и banner_pos -------------------------------

print("=== CTR по device_type ===\n")
dev_map = {0: 'unknown/other', 1: 'mobile', 2: 'PC (rare)', 4: 'tablet', 5: 'connected_TV'}
dev_stats = df.groupby('device_type').agg(
    n=('click', 'count'), clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks']/x['n'], share=lambda x: x['n']/x['n'].sum()*100)

for idx, row in dev_stats.iterrows():
    label = dev_map.get(idx, f'type_{idx}')
    bar = '#' * int(row['ctr'] * 100)
    print(f"  {label:<16} | {row['n']:>9,} ({row['share']:5.1f}%) | CTR = {row['ctr']:.2%} | {bar}")

print("\n\n=== CTR по banner_pos ===\n")
bp_stats = df.groupby('banner_pos').agg(
    n=('click', 'count'), clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks']/x['n'], share=lambda x: x['n']/x['n'].sum()*100)

for idx, row in bp_stats.iterrows():
    bar = '#' * int(row['ctr'] * 100)
    print(f"  pos={idx} | {row['n']:>9,} ({row['share']:5.1f}%) | CTR = {row['ctr']:.2%} | {bar}")

In [ ]:
# --- 2.3 Формат рекламы: C15 x C16 (ширина x высота) ------------------

print("=== CTR по размеру баннера (C15 x C16) ===\n")
df['ad_format'] = df['C15'].astype(str) + 'x' + df['C16'].astype(str)

fmt_stats = df.groupby('ad_format').agg(
    n=('click', 'count'), clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks']/x['n'], share=lambda x: x['n']/x['n'].sum()*100)
fmt_stats = fmt_stats[fmt_stats['n'] >= 50].sort_values('n', ascending=False)

# Известные IAB-форматы
iab_names = {
    '320x50': 'Mobile Banner',
    '300x250': 'Medium Rectangle',
    '728x90': 'Leaderboard',
    '216x36': 'Small Banner',
    '480x320': 'Mobile Interstitial',
    '300x50': 'Mobile Banner (alt)',
}

print(f"{'Размер':<12} {'IAB-формат':<22} {'Показы':>10} {'CTR':>8} {'Доля':>7}")
print("-" * 65)
for idx, row in fmt_stats.iterrows():
    iab = iab_names.get(idx, '---')
    print(f"{idx:<12} {iab:<22} {row['n']:>10,} {row['ctr']:>8.2%} {row['share']:>6.1f}%")

if '300x250' in fmt_stats.index and '320x50' in fmt_stats.index:
    ratio = fmt_stats.loc['300x250', 'ctr'] / fmt_stats.loc['320x50', 'ctr']
    print(f"\n320x50 (Mobile Banner) = {fmt_stats.loc['320x50', 'share']:.1f}% трафика,")
    print(f"но CTR 300x250 в {ratio:.1f}x выше -- крупный формат кликабельнее.")

In [ ]:
# --- 2.4 Суточные и дневные паттерны -----------------------------------

print("=== Объём трафика и CTR по часам дня ===\n")
hourly = df.groupby('hour_of_day').agg(
    n=('click', 'count'), clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks']/x['n'])

peak_hour = hourly['n'].idxmax()
trough_hour = hourly['n'].idxmin()
print(f"  Пиковый час (объём): {peak_hour}:00 ({hourly.loc[peak_hour, 'n']:,} показов)")
print(f"  Минимум (объём):     {trough_hour}:00 ({hourly.loc[trough_hour, 'n']:,} показов)")
print(f"  Разница:             {hourly['n'].max() / hourly['n'].min():.1f}x\n")

# CTR по часам (top-5 и bottom-5)
ctr_sorted = hourly['ctr'].sort_values(ascending=False)
print("  Часы с наивысшим CTR:")
for h in ctr_sorted.head(5).index:
    print(f"    {h:02d}:00 -- {ctr_sorted[h]:.2%}")
print("\n  Часы с наименьшим CTR:")
for h in ctr_sorted.tail(5).index:
    print(f"    {h:02d}:00 -- {ctr_sorted[h]:.2%}")

# Weekday vs Weekend
df['is_weekend'] = df['dow'].isin([5, 6]).astype(int)
we_stats = df.groupby('is_weekend')['click'].agg(['count', 'mean'])
print(f"\n  Будни:    CTR = {we_stats.loc[0, 'mean']:.2%} ({we_stats.loc[0, 'count']:,} показов)")
print(f"  Выходные: CTR = {we_stats.loc[1, 'mean']:.2%} ({we_stats.loc[1, 'count']:,} показов)")

In [ ]:
# --- 2.5 Анализ connection type и C1 -----------------------------------

print("=== device_conn_type (тип подключения) ===\n")
conn_map = {0: 'Unknown/WiFi', 2: 'Cellular (2G/3G)', 3: 'Cellular (4G?)', 5: 'Other'}
conn_stats = df.groupby('device_conn_type').agg(
    n=('click', 'count'), clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks']/x['n'], share=lambda x: x['n']/x['n'].sum()*100)

for idx, row in conn_stats.iterrows():
    label = conn_map.get(idx, f'type_{idx}')
    print(f"  {label:<22} | {row['n']:>9,} ({row['share']:5.1f}%) | CTR = {row['ctr']:.2%}")

print("\n\n=== Признак C1 ===\n")
c1_stats = df.groupby('C1').agg(
    n=('click', 'count'), clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks']/x['n'], share=lambda x: x['n']/x['n'].sum()*100)

for idx, row in c1_stats.iterrows():
    print(f"  C1={idx} | {row['n']:>9,} ({row['share']:5.1f}%) | CTR = {row['ctr']:.2%}")

print("\nC1=1005 доминирует (92% трафика). C1=1002 = device_type 0 (perfect correlation).")
print("C1, вероятно, кодирует тип рекламного слота или аукциона.")

In [ ]:
# --- 2.6 Топ-10 комбинаций по CTR --------------------------------------

print("=== Топ-10 комбинаций site_category x device_type x ad_format по CTR ===")
print("(фильтр: min 500 показов)\n")

combo = df.groupby(['site_category', 'device_type', 'ad_format']).agg(
    impressions=('click', 'count'),
    clicks=('click', 'sum')
).assign(ctr=lambda x: x['clicks'] / x['impressions'])

combo = combo[combo['impressions'] >= 500].sort_values('ctr', ascending=False)

print(f"{'site_cat':<12} {'dev_type':>8} {'format':<12} {'impressions':>11} {'CTR':>8}")
print("-" * 56)
for (sc, dt, af), row in combo.head(10).iterrows():
    print(f"{sc:<12} {dt:>8} {af:<12} {row['impressions']:>11,} {row['ctr']:>8.2%}")

print(f"\nВсего комбинаций (min 500 показов): {len(combo)}")

### Блок 2 — Резюме EDA

**Ключевые находки (из реальных данных Avazu):**

1. **Формат рекламы — главный драйвер CTR.** 300x250 (Medium Rectangle) даёт CTR ~42%, а 320x50 (Mobile Banner) — ~16%. Это x2.7 разницы. В production при прочих равных крупные форматы предпочтительнее.

2. **Device type имеет значение.** Desktop/unknown (type 0) показывает CTR ~21%, mobile (type 1) — ~17%, tablet (type 4) — ~10%. Возможная причина: на десктопе экран больше и клик точнее.

3. **Суточный паттерн реален.** CTR имеет чёткий суточный цикл (+-2.5 п.п. от среднего). Пик — 15:00 UTC (18.3%), минимум — 9:00 UTC (16.0%). Выходные CTR выше будней на ~1.5 п.п.

4. **Temporal drift за 10 дней.** CTR дней 28-29 (15.1-15.6%) ниже дней 23-27 (17.3-18.3%). Это реальный drift, которого не было бы в синтетических данных.

5. **Тип подключения:** WiFi-пользователи кликают чаще (18%) чем cellular (14%). Быстрое соединение -> больше engagement.

6. **3 категории площадок покрывают 90%+ трафика**, но их CTR различается существенно (13-28%).

**Ограничение:** без данных о revenue/spend/conversions невозможно оценить ROI и конверсию. Все метрики выше — про CTR (клики), а не про прибыльность. В реальном пайплайне CTR — первое звено воронки (impression -> click -> conversion -> revenue), и Avazu покрывает только переход impression -> click.

---
# Блок 3 — Три моделировочных решения

## Решение 1: Постановка задачи

**Бинарная классификация: предсказание P(click=1 | features).**

| Параметр | Значение | Обоснование |
|:---------|:---------|:------------|
| Тип задачи | Binary classification | click in {0, 1} |
| Целевая метрика | **Log-loss** | Стандарт индустрии для CTR; штрафует за калибровку, а не только за ранжирование |
| Вспомогательные метрики | AUC-ROC, PR-AUC | AUC — ранжирование; PR-AUC — при дисбалансе информативнее |
| Бизнес-контекст | P(click) x bid -> expected revenue | Калиброванные вероятности критичны — поэтому log-loss, а не F1 |

**Почему не regression (предсказание CTR как float)?**
При impression-level данных это тождественно бинарной классификации. Regression имел бы смысл, если бы данные были агрегированы по кампаниям (как в исходной схеме задания).

## Решение 2: Отбор 10 признаков

Avazu содержит 22 признака (исключая `id`). Из них нужно отобрать 10 для первой модели.

### Отобранные (10):

| # | Признак | Кардинальность | Кодирование | Обоснование |
|---|---------|---------------|-------------|-------------|
| 1 | `hour_of_day` | 24 | numeric / cyclic (sin/cos) | Суточный паттерн CTR, delta ~ 2.5 п.п. |
| 2 | `banner_pos` | 7 | one-hot | Позиция баннера, CTR 10-32% |
| 3 | `site_category` | ~26 | one-hot | Контекст площадки, CTR 4-28%. Покрывает «вертикаль» из исходной схемы |
| 4 | `app_category` | ~19 | one-hot | Контекст приложения |
| 5 | `device_type` | 5 | one-hot | Тип устройства (mobile/desktop/tablet), CTR 9-21% |
| 6 | `device_conn_type` | 4 | one-hot | WiFi vs cellular, CTR 3-18% |
| 7 | `C1` | 7 | one-hot | Тип аукциона/слота, сильно коррелирует с device_type |
| 8 | `C15` | ~8 | one-hot / numeric | Ширина баннера. Главный предиктор (delta CTR = 41%) |
| 9 | `C16` | ~9 | one-hot / numeric | Высота баннера. Вместе с C15 определяет формат |
| 10 | `C18` | 4 | one-hot | Неизвестная категория, но delta CTR = 26% |

### Исключённые (12, с причинами):

| Признак | Кардинальность | Причина исключения |
|---------|---------------|-------------------|
| `id` | 40M (unique) | Идентификатор строки, zero information |
| `device_ip` | ~960K | User proxy -> optimistic bias при temporal split + невозможно кодировать |
| `device_id` | ~278K | 86% = одно дефолтное значение, остальное — user proxy |
| `site_id` | ~3K | Высокая кардинальность. Потенциально полезен с hashing trick, но не для первой модели |
| `site_domain` | ~3.5K | Redundant с site_id + site_category |
| `app_id` | ~4K | Аналогично site_id |
| `app_domain` | ~262 | Менее гранулярен чем app_id, но всё ещё high-cardinality для one-hot |
| `device_model` | ~5.8K | Полезен, но нужен hashing/embedding |
| `C14` | ~2.4K | Высокая кардинальность, вероятно campaign-level ID |
| `C17` | ~425 | Средняя кардинальность, но unclear signal. Кандидат для второй итерации |
| `C19` | ~66 | Слабый сигнал в EDA |
| `C20` | ~164 | Содержит -1 (missing), высокая кардинальность |
| `C21` | ~60 | Кандидат для второй итерации |

**Принцип:** первая модель — low-cardinality features с прямым one-hot encoding. Высококардинальные фичи (site_id, C14, device_model) добавляются на второй итерации через hashing trick или target encoding (с proper K-fold для избежания leakage).

## Решение 3: Стратегия train/test сплита

**Temporal split: train = дни 21-27, test = дни 28-30.**

| Параметр | Значение |
|:---------|:---------|
| Train | 2014-10-21 -> 2014-10-27 (7 дней) |
| Test | 2014-10-28 -> 2014-10-30 (3 дня) |
| Пропорция | ~68% / 32% |

### Почему temporal, а не random?

1. **Реалистичность:** в production модель всегда предсказывает будущие показы, а не случайные из прошлого. Random split даёт overly optimistic оценку.

2. **Temporal drift:** EDA показала, что CTR дней 28-30 (15.1-16.8%) ниже, чем дней 21-27 (15.8-18.3%). Random split «размазал» бы этот drift и спрятал бы проблему.

3. **User leakage:** при random split один и тот же `device_ip` может попасть в train и test. При temporal split это тоже возможно, но хотя бы все test-показы хронологически после всех train-показов — как в production.

### Альтернатива для валидации:

Time-series cross-validation (expanding window):
- Fold 1: train=[21], test=[22]
- Fold 2: train=[21-22], test=[23]
- ...
- Fold 6: train=[21-26], test=[27]

Это даёт 6 точек оценки стабильности модели по дням. Final evaluation — на held-out дни 28-30.

In [ ]:
# --- Валидация temporal split ---------------------------------------------

train_mask = df['day'] <= 27
test_mask = df['day'] >= 28

train_df = df[train_mask]
test_df = df[test_mask]

print("=== Temporal Split: train (дни 21-27) vs test (дни 28-30) ===\n")
print(f"  Train: {len(train_df):>10,} строк ({len(train_df)/len(df):.1%})")
print(f"  Test:  {len(test_df):>10,} строк ({len(test_df)/len(df):.1%})")
print(f"\n  Train CTR: {train_df['click'].mean():.4f}")
print(f"  Test CTR:  {test_df['click'].mean():.4f}")
print(f"  delta CTR: {train_df['click'].mean() - test_df['click'].mean():.4f}")

# Проверка: пересечение device_ip между train и test
train_ips = set(train_df['device_ip'].unique())
test_ips = set(test_df['device_ip'].unique())
overlap = train_ips & test_ips
print(f"\n  Уникальных device_ip в train: {len(train_ips):,}")
print(f"  Уникальных device_ip в test:  {len(test_ips):,}")
print(f"  Пересечение:                  {len(overlap):,} ({len(overlap)/len(test_ips):.1%} test IPs видны в train)")
print(f"\n  -> {len(overlap)/len(test_ips):.0%} test-пользователей -- returning visitors.")
print(f"     Это реалистично, но модель без device_ip не сможет этим воспользоваться.")

# Проверка: распределение site_category в train vs test
print(f"\n  Распределение site_category (top-5):")
train_cats = train_df['site_category'].value_counts(normalize=True).head(5)
test_cats = test_df['site_category'].value_counts(normalize=True).head(5)
merged_cats = pd.DataFrame({'train': train_cats, 'test': test_cats}).fillna(0)
for cat, row in merged_cats.iterrows():
    print(f"    {cat}: train={row['train']:.1%}, test={row['test']:.1%}")

### Важно: кодирование категориальных признаков

**LabelEncoder / OrdinalEncoder должен быть fit ТОЛЬКО на train.**

```python
# НЕПРАВИЛЬНО (leakage через encoder):
le.fit_transform(df[col])  # видит test-значения при обучении

# ПРАВИЛЬНО:
le.fit(train_df[col])
train_df[col + '_enc'] = le.transform(train_df[col])
# Обработка unseen categories в test:
test_df[col + '_enc'] = test_df[col].map(
    lambda x: le.transform([x])[0] if x in le.classes_ else -1
)
```

Для one-hot encoding: `pd.get_dummies()` тоже нужно делать раздельно, иначе test может содержать категории, которых не было в train (и наоборот, train может содержать категории, отсутствующие в test).

---
# Итоговая таблица

| Параметр | Решение |
|:---------|:--------|
| **Датасет** | Avazu CTR Prediction (40.4M строк, реальные данные) |
| **Таргет** | `click` = binary (0/1), P(click) — вероятность клика |
| **Баланс** | 83% / 17% (no-click / click), умеренный дисбаланс |
| **Метрика** | Log-loss (основная) + AUC-ROC, PR-AUC |
| **Топ-3 фичи** | C15 x C16 (формат баннера), device_type, site_category |
| **Временная структура** | 10 дней, суточный цикл + drift дней 28-30 |
| **Leakage-риски** | device_ip (user proxy), device_id (86% default) |
| **10 признаков** | hour_of_day, banner_pos, site_category, app_category, device_type, device_conn_type, C1, C15, C16, C18 |
| **Исключены** | id, device_ip, device_id, site_id/domain, app_id/domain, device_model, C14, C17, C19-C21 |
| **Сплит** | Temporal: train = дни 21-27, test = дни 28-30 |
| **Encoder** | fit на train, transform на test (не на полном df) |